In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import steervla_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

In [4]:
# Load the model 
config = _config.get_config("pi05_steervla_cot_ki")
checkpoint_path = "gs://cat-logs/pi05_steervla_cot_ki/pi05_steervla_cot_ki/90000"
checkpoint_dir = download.maybe_download(checkpoint_path)

In [7]:
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

In [8]:
example = steervla_policy.make_steervla_example()
result = policy.infer_with_cot(example)

AttributeError: 'Policy' object has no attribute 'infer_with_cot'

In [9]:
result

{'actions': array([[0.55259805, 0.20640477, 0.9384949 , 0.30594815],
        [0.55305294, 0.15525594, 0.94101202, 0.31205152],
        [0.57089067, 0.07142587, 0.94443476, 0.30343889],
        [0.59314093, 0.04531255, 0.94652192, 0.30325871],
        [0.61630757, 0.10363536, 0.91389583, 0.40010151],
        [0.61505153, 0.17083294, 0.93075266, 0.35193466],
        [0.57699193, 0.26548221, 0.95733383, 0.27394587],
        [0.5801798 , 0.25804765, 0.96886617, 0.21244308],
        [0.61121951, 0.18985562, 0.97188809, 0.18826736],
        [0.61235117, 0.13615773, 0.97071061, 0.20000983]]),
 'policy_timing': {'infer_ms': 66.27605599351227}}

In [29]:

del model

NameError: name 'model' is not defined

In [4]:
# Now load examples from the dataset

config = dataclasses.replace(config, batch_size=2)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

In [38]:
config = dataclasses.replace(config, data=dataclasses.replace(config.data, dataset_name_weight_mappings={"simlingo_dataset_all_img512_1116": 1.0}))
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

2026-04-13 17:59:37.350380: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [39]:
import sentencepiece
import openpi.shared.download as download
import openpi.models.tokenizer as tokenizer

tokenizer = tokenizer.CoTPaligemmaTokenizer()

sp = tokenizer._tokenizer


In [40]:
import numpy as np
def _decode_tokens(token_ids: np.ndarray, mask: np.ndarray, tokenizer) -> str:
    """Decode a padded token array back to text using the given SP tokenizer."""
    valid = token_ids[mask.astype(bool)]
    return tokenizer.decode(valid.tolist())

In [ ]:
token = tokenizer.vocab_size - 1 - 128 - 1 
tokenizer.
_decode_tokens(np.array([token]), np.array([True]), tokenizer._tokenizer)

'<loc1022>'

In [34]:
start_subtask_text = _decode_tokens(np.array([tokenizer._start_of_subtask()]), np.array([True]), tokenizer._tokenizer)
end_subtask_text = _decode_tokens(np.array([tokenizer._end_of_subtask()]), np.array([True]), tokenizer._tokenizer)
start_reasoning_text = _decode_tokens(np.array([tokenizer._start_of_reasoning()]), np.array([True]), tokenizer._tokenizer)
end_reasoning_text = _decode_tokens(np.array([tokenizer._end_of_reasoning()]), np.array([True]), tokenizer._tokenizer)
example_prompt = "Prompt:A car is driving down the street;State:84 116;" + start_subtask_text + "Follow the road normally;" + end_subtask_text + start_reasoning_text + "The car is driving down the street;" + end_reasoning_text
print(example_prompt)
subtask = example_prompt.split(end_subtask_text)[0].split(start_subtask_text)[1]
reasoning = example_prompt.split(end_reasoning_text)[0].split(start_reasoning_text)[1]
print(subtask)
print(reasoning)
print(tokenizer._tokenizer.eos_id())

Prompt:A car is driving down the street;State:84 116;<loc1022>Follow the road normally;<loc1021><loc1020>The car is driving down the street;<loc1019>
Follow the road normally;
The car is driving down the street;
1


In [43]:
string = " blahblahblah"
string.split(";")[0]


' blahblahblah'

In [12]:
prompt = _decode_tokens(obs.tokenized_prompt[0], obs.tokenized_prompt_mask[0], sp)
subtask = _decode_tokens(obs.tokenized_subtask[0], obs.tokenized_subtask_mask[0], sp)
reasoning = _decode_tokens(obs.tokenized_reasoning[0], obs.tokenized_reasoning_mask[0], sp)

In [13]:
print("Prompt: ", prompt)
print("Subtask: ", subtask)
print("Reasoning: ", reasoning)


Prompt:  Prompt:The current speed is 8.225407600402832 m/s. go right at the next intersection in 12 meter then do a lane change to the right.|State:180 149|Subtask:
Subtask:  The car is in a driving scenario.|Reasoning:
Reasoning:  Follow the route.|


In [41]:
key = jax.random.key(0)
cot = model.sample_cot(key, obs)

NameError: name 'model' is not defined

In [17]:
detokenized_cot = _decode_tokens(cot["tokenized_subtask"][0], cot["tokenized_subtask_mask"][0], sp)

In [18]:
detokenized_cot

'. vehicle turns right, accelerating normally through the junction.|50.0 mps. Accelerate to drive through the junction.||Turn right. Accelerate to drive through the junction.|||Turn right. Accelerate'

In [22]:

import logging

import einops
import flax.nnx as nnx
import flax.nnx.bridge as nnx_bridge
import jax
import jax.numpy as jnp
from typing_extensions import override

from openpi.models import model as _model
from openpi.models import pi0
from openpi.models import pi0_config
import openpi.models.gemma as _gemma
import openpi.models.siglip as _siglip
from openpi.shared import array_typing as at

max_subtask_len = 24


observation = _model.preprocess_observation(None, obs, train=False)
if observation.tokenized_prompt is None or observation.tokenized_prompt_mask is None:
    raise ValueError("sample_cot requires tokenized_prompt and tokenized_prompt_mask")

batch_size = observation.state.shape[0]
ms = model.max_subtask_len
mr = model.max_reasoning_len

In [23]:
print(ms)
print(mr)

48
96


In [ ]:
# Embed images
img_tokens, img_masks, img_ar = model._embed_images(observation)
        
# Embed prompt
prompt_emb = model._embed_text_tokens(observation.tokenized_prompt)
prompt_mask = observation.tokenized_prompt_mask
        
# Construct prefix
prefix_tokens = jnp.concatenate(img_tokens + [prompt_emb], axis=1)
prefix_mask = jnp.concatenate(img_masks + [prompt_mask], axis=1)
prefix_ar = jnp.array(img_ar + [False] * prompt_emb.shape[1])
        
# Build attention mask
prefix_attn_mask = pi0.make_attn_mask(prefix_mask, prefix_ar)
positions = jnp.cumsum(prefix_mask, axis=1) - 1

# Prefill prefix
(prefix_out, _), kv_cache = model.PaliGemma.llm(
    [prefix_tokens, None], mask=prefix_attn_mask, positions=positions
)
assert prefix_out is not None

In [36]:
prefix_tokens[0].shape

(880, 2048)

In [29]:
rng = jax.random.key(0)
h = model._gather_last_valid_hidden(prefix_out, prefix_mask)
key_mask = prefix_mask
abs_pos = jnp.sum(prefix_mask, axis=1, keepdims=True).astype(jnp.int32)

# Buffer for subtask
sub_buf = jnp.zeros((batch_size, ms), dtype=jnp.int32)
sub_m = jnp.zeros((batch_size, ms), dtype=jnp.bool_)
rng_cur = rng

In [37]:
print(_decode_tokens(observation.tokenized_prompt[0], observation.tokenized_prompt_mask[0], sp))

Prompt:The current speed is 8.225407600402832 m/s. go right at the next intersection in 12 meter then do a lane change to the right.|State:180 149|Subtask:


In [51]:
print(sp.encode("|")[0])

235371


In [54]:
temperature = 0.9
# Generate subtask
for i in range(ms):
    print("iter: ", i)
    # Decode logits
    logits = model.PaliGemma.llm(h, method="decode_logits")[:, 0, :]
    if temperature and temperature > 0:
        # Temperature sampling
        rng_cur, step_rng = jax.random.split(rng_cur)
        tok = jax.random.categorical(step_rng, logits / jnp.maximum(temperature, 1e-6))
    else:
        # Greedy
        tok = jnp.argmax(logits, axis=-1)
    
    # Update buffer
    sub_buf = sub_buf.at[:, i].set(tok)
    sub_m = sub_m.at[:, i].set(True)
    if i == ms - 1:
        break
    
    # Embed token
    emb = model._embed_text_tokens(tok[:, None])
    key_mask = jnp.concatenate([key_mask, jnp.ones((batch_size, 1), dtype=jnp.bool_)], axis=1)
    attn_mask = key_mask[:, None, :]  # (b, 1, k_len); Module adds head axis
    (out, _), kv_cache = model.PaliGemma.llm(
        [emb, None],
        mask=attn_mask,
        positions=abs_pos,
        kv_cache=kv_cache,
    )
    assert out is not None
    h = out
    abs_pos = abs_pos + 1

iter:  0
iter:  1
iter:  2
iter:  3
iter:  4
iter:  5
iter:  6
iter:  7
iter:  8
iter:  9
iter:  10
iter:  11
iter:  12
iter:  13
iter:  14
iter:  15
iter:  16
iter:  17
iter:  18
iter:  19
iter:  20
iter:  21
iter:  22
iter:  23
iter:  24
iter:  25
iter:  26
iter:  27
iter:  28
iter:  29
iter:  30
iter:  31
iter:  32
iter:  33
iter:  34
iter:  35
iter:  36
iter:  37
iter:  38
iter:  39
iter:  40
iter:  41
iter:  42
iter:  43
iter:  44
iter:  45
iter:  46
iter:  47


In [44]:
subtask = _decode_tokens(sub_buf[0], sub_m[0], sp)
print(subtask)

0 and200 mps while cautiously turning right, accelerated to speed limit, and making a wide adjustment.| an bicycle.|6. with a wide, steady gray car.|5. Decelerate to stay behind the


In [ ]:
rea_buf = jnp.zeros((batch_size, mr), dtype=jnp.int32)
rea_m = jnp.zeros((batch_size, mr), dtype=jnp.bool_)

for j in range(mr):
    logits = self.PaliGemma.llm(h, method="decode_logits")[:, 0, :]
    if temperature and temperature > 0:
    rng_cur, step_rng = jax.random.split(rng_cur)
    tok = jax.random.categorical(step_rng, logits / jnp.maximum(temperature, 1e-6))
else:
    tok = jnp.argmax(logits, axis=-1)
rea_buf = rea_buf.at[:, j].set(tok)
rea_m = rea_m.at[:, j].set(True)
if j == mr - 1:
    break
emb = self._embed_text_tokens(tok[:, None])
key_mask = jnp.concatenate([key_mask, jnp.ones((batch_size, 1), dtype=jnp.bool_)], axis=1)
attn_mask = key_mask[:, None, :]
(out, _), kv_cache = self.PaliGemma.llm(
    [emb, None],
    mask=attn_mask,
    positions=abs_pos,
    kv_cache=kv_cache,
)
assert out is not None
h = out
abs_pos = abs_pos + 1

In [13]:
# Now we can sample actions and cot from the model
key = jax.random.key(0)
actions = model.sample_actions(key, obs, num_steps=10)
cot = model.sample_cot(key, obs)

In [14]:
actions

Array([[[ 3.88848573e-01,  6.02547526e-02,  1.00179613e+00,
          2.07083803e-02, -4.20849770e-04,  2.88769603e-04,
         -7.31848180e-04,  2.94964761e-04, -7.81916082e-04,
         -4.89652157e-05, -1.54981017e-03,  3.62396240e-05,
          1.14153698e-03,  4.66899946e-04, -1.56873465e-03,
          1.06370449e-03,  3.11493874e-04,  2.76226550e-04,
          9.86610306e-04, -3.03566456e-04,  1.63629651e-03,
         -3.37496400e-04, -1.26321428e-03, -9.18112695e-04,
         -1.40171219e-03,  2.53662467e-04,  1.14985555e-03,
          1.37202442e-04,  5.17979264e-04,  1.21071935e-05,
         -1.06802583e-03,  9.69402492e-04],
        [ 3.54255378e-01,  2.86250532e-01,  1.00884748e+00,
          1.60496011e-02,  5.73627651e-04,  7.74785876e-05,
         -1.10794720e-03,  3.98746692e-04, -1.85314566e-03,
          3.93435359e-04, -4.41454351e-04, -1.02818012e-05,
          2.17931345e-04, -8.21486115e-04, -5.74216247e-04,
         -8.94561410e-04,  2.49862671e-04,  4.76896763e-

In [ ]:
cot

{'tokenized_subtask': Array([[235265,   8692,  12722,   1833, 235269,  83966,  16342,   1593,
            573,  36384, 235265, 235371, 235308, 235276, 235265, 235276,
            519,    833, 235265,  48952,    607,    577,   6109,   1593,
            573,  36384, 235265, 235371, 235371,  17208,   1833, 235265,
          48952,    607,    577,   6109,   1593,    573,  36384, 235265,
         235371, 235371, 235371,  17208,   1833, 235265,  48952,    607],
        [235265,   8692,  43494, 135727,   3205,    577,    573,   1833,
           2183,  16342,  22522,   4969,   2449,    573,   6584,   3440,
         235265, 235371,   4929,   2449,    573,   6584,   3440, 235265,
           4809,   6972,    607,    577,   6109,    675,    573,   4408,
           4969, 235265, 235371,   4929,   2449,    573,   6584,   3440,
         235265,   4809,   6972,    607,    577,   6109,    675,    573]],      dtype=int32),
 'tokenized_subtask_mask': Array([[ True,  True,  True,  True,  True,  True,  Tru

In [ ]:
print("Prompt: ", _decode_tokens(obs.tokenized_prompt, obs.tokenized_prompt_mask, sp))

Prompt:  Prompt:The current speed is 8.225407600402832 m/s. go right at the next intersection in 12 meter then do a lane change to the right.|State:180 149|Subtask:Prompt:The current speed is 6.162538528442383 m/s. follow the road.|State:167 125|Subtask:


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Now detokenize the actions and cot


In [21]:
detokenized_cot

'. vehicle turns right, accelerating normally through the junction.|50.0 mps. Accelerate to drive through the junction.||Turn right. Accelerate to drive through the junction.|||Turn right. Accelerate. vehicle smoothly adjusts course to the right while normally maintaining speed around the construction site.|Go around the construction site. Decelerate to drive with the target speed.|Go around the construction site. Decelerate to drive with the'